<a href="https://colab.research.google.com/github/Prashkov1ch/python-ai-Prashkovich-Anna/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_Lab04_Economists_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 4 (экономисты): Pandas — Series, DataFrame, Index, пропуски, MultiIndex

**Ограничения:** упор на `pandas`, частично `numpy`.

**Цель:** отработать:
- объекты Series/DataFrame/Index,
- индексацию и выборку,
- операции над данными,
- обработку пропусков,
- MultiIndex.

In [3]:
import re
from openpyxl import load_workbook
import numpy as np
import os
from google.colab import drive

# 1. Монтируем диск (если уже смонтирован, просто игнорируем ошибку)
try:
    drive.mount('/content/drive')
except:
    pass

PATH = '/content/drive/MyDrive/цифровая кафедра/'  # Проверь, что имя папки верное!
os.chdir(PATH)

FILES = ['Копия 1 Атомэнергопром.xlsx', 'Копия 2 Аэрофлот.xlsx', 'Копия 3 Газпром_петрозаводск.xlsx',
         'Копия 4 Лукойл.xlsx', 'Копия 5 Роснефть.xlsx', 'Копия 6 Самолет.xlsx',
         'Копия 7 Славмо.xlsx', 'Копия 8 Строительная_компания_Век.xlsx',
         'Копия 9 ТГК_1.xlsx', 'Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx']

# --- ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ОЧИСТКИ ЧИСЕЛ ---
def clean_number(value):
    """
    Превращает значение из Excel в число float.
    Удаляет пробелы из строк вида '1 000 000'.
    Если значение пустое — возвращает np.nan.
    """
    if value is None:
        return np.nan
    # Превращаем в строку и убираем все пробелы
    s = str(value).replace(' ', '')
    try:
        return float(s)
    except:
        return np.nan

def org_info(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Сведения об организации"]
    return {
        "file": xlsx_name,
        "company": str(ws.cell(6, 13).value),
        "inn": str(ws.cell(10, 13).value),
        "okved": str(ws.cell(15, 13).value),
        "address": str(ws.cell(16, 13).value),
    }

def parse_financial(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Отчет о финансовых результатах"]
    res = {}
    for r in range(6, 200):
        code = ws.cell(r, 16).value
        if code is None:
            continue
        code = str(code).strip()
        if not re.fullmatch(r"\d{4}", code):
            continue
        v23 = ws.cell(r, 21).value
        v22 = ws.cell(r, 27).value
        # ИСПРАВЛЕНО: используем clean_number для очистки данных
        res[code] = {2022: clean_number(v22), 2023: clean_number(v23)}
    return res

def parse_balance(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Бухгалтерский баланс"]
    res = {}
    for r in range(6, 400):
        code = ws.cell(r, 14).value
        if code is None:
            continue
        code = str(code).strip()
        if not re.fullmatch(r"\d{4}", code):
            continue
        v23 = ws.cell(r, 17).value
        v22 = ws.cell(r, 23).value
        v21 = ws.cell(r, 30).value
        # ИСПРАВЛЕНО: используем clean_number для очистки данных
        res[code] = {2021: clean_number(v21), 2022: clean_number(v22), 2023: clean_number(v23)}
    return res

rows = []
for fn in FILES:
    org = org_info(fn)
    fin = parse_financial(fn)
    bal = parse_balance(fn)

    for year in [2022, 2023]:
        rows.append({
            "company": org["company"],
            "file": org["file"],
            "inn": org["inn"],
            "okved": org["okved"],
            "address": org["address"],
            "year": year,
            "revenue_2110": fin.get("2110", {}).get(year, np.nan),
            "net_profit_2400": fin.get("2400", {}).get(year, np.nan),
            "assets_1600": bal.get("1600", {}).get(year, np.nan),
            "equity_1300": bal.get("1300", {}).get(year, np.nan),
        })

df = pd.DataFrame(rows)
print("✅ Данные успешно загружены и очищены!")
print(f"Shape: {df.shape}")
print(f"Компаний: {df['company'].nunique()}")
print(f"Годы: {sorted(df['year'].unique())}")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ Данные успешно загружены и очищены!
Shape: (20, 10)
Компаний: 8
Годы: [np.int64(2022), np.int64(2023)]


,company,file,inn,okved,address,year,revenue_2110,net_profit_2400,assets_1600,equity_1300
0,"АКЦИОНЕРНОЕ ОБЩЕСТВО ""АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕНН...",Копия 1 Атомэнергопром.xlsx,7706664260,62.09,"119017, Москва г, ул Ордынка Б., 24",2022,1851540.0,NaN,2.222861e+09,1.464189e+09
1,"АКЦИОНЕРНОЕ ОБЩЕСТВО ""АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕНН...",Копия 1 Атомэнергопром.xlsx,7706664260,62.09,"119017, Москва г, ул Ордынка Б., 24",2023,10667978.0,91493932.0,2.275888e+09,1.516195e+09
2,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""АЭРОФЛОТ-РОССИ...",Копия 2 Аэрофлот.xlsx,7712040126,51.10.1,"119019, Москва г, ул Арбат, 1",2022,332747813.0,NaN,7.847067e+08,NaN
3,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""АЭРОФЛОТ-РОССИ...",Копия 2 Аэрофлот.xlsx,7712040126,51.10.1,"119019, Москва г, ул Арбат, 1",2023,497511315.0,NaN,9.342681e+08,NaN
4,"АКЦИОНЕРНОЕ ОБЩЕСТВО ""ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕН...",Копия 3 Газпром_петрозаводск.xlsx,1001009551,35.22,"185011, Карелия Респ, Петрозаводск г, Балтийск...",2022,NaN,NaN,NaN,NaN


### Задание 1
Создайте `Series` выручки за 2023 год, где индекс — `company`, значения — `revenue_2110`.  
Покажите `index`, `dtype` и первые 5 элементов.

In [4]:
# Задание 1: Создание Series выручки за 2023 год

# 1. Фильтруем данные только за 2023 год
# Создаём булеву маску: year == 2023
mask_2023 = df['year'] == 2023

# 2. Создаём Series с выручкой за 2023 год
# index=df.loc[mask_2023, 'company'] — названия компаний как индекс
# values=df.loc[mask_2023, 'revenue_2110'] — значения выручки
revenue_2023_series = pd.Series(
    data=df.loc[mask_2023, 'revenue_2110'].values,
    index=df.loc[mask_2023, 'company'].values
)

# 3. Показываем индекс Series
print("📋 Индекс Series (названия компаний):")
print(revenue_2023_series.index)

# 4. Показываем тип данных (dtype)
print("\n📊 Тип данных (dtype):")
print(revenue_2023_series.dtype)

# 5. Показываем первые 5 элементов
print("\n📈 Первые 5 элементов Series:")
print(revenue_2023_series.head())

# 6. Дополнительная информация для понимания
print("\n" + "=" * 50)
print("Дополнительная информация:")
print("=" * 50)
print(f"Длина Series: {len(revenue_2023_series)}")
print(f"Тип объекта: {type(revenue_2023_series)}")

📋 Индекс Series (названия компаний):
Index(['АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"',
       'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"',
       ' АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"',
       'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"',
       ' АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"',
       'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ"',
       'АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО"',
       'АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК"',
       'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1"',
       'АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"'],
      dtype='object')

📊 Тип данных (dtype):
float64

📈 Первые 5 элементов Series:
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"        1.066798e+07
ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"    4.975113e+08
 АКЦИО

### Задание 2
Проверьте, какие компании имеют пропуски (NaN) по `net_profit_2400` в 2023.  
Сделайте булеву маску и выведите список названий компаний.

In [5]:
# Задание 2: Поиск компаний с пропусками в net_profit_2400 за 2023 год

# 1. Создаём булеву маску для фильтрации данных за 2023 год
mask_2023 = df['year'] == 2023

# 2. Создаём булеву маску для поиска пропусков (NaN) в столбце net_profit_2400
# pd.isna() возвращает True для значений NaN, False для остальных
mask_nan = pd.isna(df['net_profit_2400'])

# 3. Объединяем две маски с помощью логического И (&)
# Нам нужны строки, где год = 2023 И значение прибыли = NaN
mask_missing_profit = mask_2023 & mask_nan

# 4. Используем маску для выбора названий компаний с пропусками
# df.loc[маска, 'column'] — выборка по маске из указанного столбца
companies_with_nan = df.loc[mask_missing_profit, 'company']

# 5. Преобразуем результат в список для удобного вывода
companies_list = companies_with_nan.tolist()

# 6. Выводим результат
print("🔍 Компании с пропуском (NaN) в net_profit_2400 за 2023 год:")
print("-" * 60)

if len(companies_list) == 0:
    print("✅ Все компании имеют данные о чистой прибыли за 2023 год!")
else:
    print(f"Найдено компаний с пропусками: {len(companies_list)}\n")
    for i, company in enumerate(companies_list, 1):
        print(f"  {i}. {company}")

# 7. Дополнительная информация: общая статистика
print("\n" + "=" * 60)
print("📊 Дополнительная статистика:")
print("=" * 60)
total_2023 = mask_2023.sum()  # количество строк за 2023 год
nan_count = mask_missing_profit.sum()  # количество пропусков
print(f"  Всего записей за 2023 год: {total_2023}")
print(f"  Записей с данными о прибыли: {total_2023 - nan_count}")
print(f"  Записей с пропусками (NaN): {nan_count}")
print(f"  Доля пропусков: {nan_count / total_2023 * 100:.1f}%")

🔍 Компании с пропуском (NaN) в net_profit_2400 за 2023 год:
------------------------------------------------------------
Найдено компаний с пропусками: 5

  1. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"
  2.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"
  3.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"
  4. АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО"
  5. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1"

📊 Дополнительная статистика:
  Всего записей за 2023 год: 10
  Записей с данными о прибыли: 5
  Записей с пропусками (NaN): 5
  Доля пропусков: 50.0%


### Задание 3
Выберите из DataFrame только столбцы `company, year, revenue_2110, net_profit_2400`.  
Покажите первые 7 строк.

In [6]:
# Задание 3: Выборка конкретных столбцов из DataFrame

# 1. Выбираем только нужные столбцы из DataFrame
# Используем двойные скобки [[]] для выбора нескольких столбцов
selected_columns = df[['company', 'year', 'revenue_2110', 'net_profit_2400']]

# 2. Показываем первые 7 строк с помощью метода .head(7)
print("📋 Первые 7 строк выбранных столбцов:")
print("=" * 80)
print(selected_columns.head(7))

# 3. Дополнительная информация о выбранных данных
print("\n" + "=" * 80)
print("📊 Информация о выборке:")
print("=" * 80)
print(f"Количество строк в DataFrame: {len(selected_columns)}")
print(f"Количество столбцов: {len(selected_columns.columns)}")
print(f"Названия столбцов: {list(selected_columns.columns)}")
print(f"Тип объекта: {type(selected_columns)}")

📋 Первые 7 строк выбранных столбцов:
                                             company  year  revenue_2110  \
0  АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕНН...  2022  1.851540e+06   
1  АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕНН...  2023  1.066798e+07   
2  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИ...  2022  3.327478e+08   
3  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИ...  2023  4.975113e+08   
4   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕН...  2022           NaN   
5   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕН...  2023           NaN   
6  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПА...  2022  2.874037e+09   

   net_profit_2400  
0              NaN  
1       91493932.0  
2              NaN  
3              NaN  
4              NaN  
5              NaN  
6      790120077.0  

📊 Информация о выборке:
Количество строк в DataFrame: 20
Количество столбцов: 4
Названия столбцов: ['company', 'year', 'revenue_2110', 'net_profit_2400']
Тип объекта: <class 'pandas.cor

### Задание 4
Создайте новый столбец `margin` = net_profit_2400 / revenue_2110.  
Обработайте деление на 0 и NaN так, чтобы получались NaN. Покажите 10 строк.

In [7]:
# Задание 4: Создание столбца margin (маржа)

# 1. Создаём новый столбец margin как отношение прибыли к выручке
# Простое деление: при делении на 0 или NaN результат будет inf или NaN
df['margin'] = df['net_profit_2400'] / df['revenue_2110']

# 2. Заменяем бесконечные значения (inf, -inf) на NaN
# Это происходит при делении на 0 или при наличии NaN в исходных данных
df['margin'] = df['margin'].replace([np.inf, -np.inf], np.nan)

# 3. Показываем 10 строк с ключевыми столбцами для проверки
print("📋 Первые 10 строк с рассчитанной маржой:")
print("=" * 90)
display_cols = ['company', 'year', 'revenue_2110', 'net_profit_2400', 'margin']
print(df[display_cols].head(10).to_string())

# 4. Дополнительная статистика по марже
print("\n" + "=" * 90)
print("📊 Статистика по столбцу margin:")
print("=" * 90)
print(f"  Всего записей: {len(df)}")
print(f"  Записей с рассчитанной маржой: {df['margin'].notna().sum()}")
print(f"  Записей с NaN (пропуски): {df['margin'].isna().sum()}")
print(f"  Средняя маржа (без учёта NaN): {df['margin'].mean():.2%}")
print(f"  Минимальная маржа: {df['margin'].min():.2%}")
print(f"  Максимальная маржа: {df['margin'].max():.2%}")

📋 Первые 10 строк с рассчитанной маржой:
                                                          company  year  revenue_2110  net_profit_2400    margin
0      АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"  2022  1.851540e+06              NaN       NaN
1      АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"  2023  1.066798e+07       91493932.0  8.576502
2  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"  2022  3.327478e+08              NaN       NaN
3  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"  2023  4.975113e+08              NaN       NaN
4   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"  2022           NaN              NaN       NaN
5   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"  2023           NaN              NaN       NaN
6      ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"  2022  2.874037e+09      790120077.0  0.274916
7      ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИ

### Задание 5
Отсортируйте данные за 2023 по `revenue_2110` по убыванию.  
Покажите топ-5 строк (company, revenue_2110).

In [8]:
# Задание 5: Топ-5 компаний по выручке за 2023 год

# 1. Фильтруем данные только за 2023 год
df_2023 = df[df['year'] == 2023].copy()

# 2. Сортируем по revenue_2110 по убыванию
# ascending=False — сортировка от большего к меньшему
# na_position='last' — NaN значения перемещаются в конец (не мешают сортировке)
df_sorted = df_2023.sort_values(
    by='revenue_2110',
    ascending=False,
    na_position='last'
)

# 3. Выбираем топ-5 строк и только нужные столбцы
top5_revenue = df_sorted[['company', 'revenue_2110']].head(5)

# 4. Выводим результат
print("🏆 Топ-5 компаний по выручке за 2023 год:")
print("=" * 70)
print(top5_revenue.to_string(index=False))

# 5. Дополнительная информация: форматированный вывод с разделителями тысяч
print("\n" + "=" * 70)
print("Топ-5 (отформатированный вывод):")
print("=" * 70)
for i, (_, row) in enumerate(top5_revenue.iterrows(), 1):
    revenue_val = row['revenue_2110']
    if pd.notna(revenue_val):
        print(f"  {i}. {row['company']}")
        print(f"     Выручка: {revenue_val:,.0f} руб.")
    else:
        print(f"  {i}. {row['company']}")
        print(f"     Выручка: нет данных")
    print()

🏆 Топ-5 компаний по выручке за 2023 год:
                                                                        company  revenue_2110
                     ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"  2753475003.0
                 ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"   497511315.0
ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1"   101697792.0
                     АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"    10667978.0
                     АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"    10667978.0

Топ-5 (отформатированный вывод):
  1. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"
     Выручка: 2,753,475,003 руб.

  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"
     Выручка: 497,511,315 руб.

  3. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1"
     Выручка: 101,697,792 руб.

  4. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫ

### Задание 6
Используя `Index`, получите пересечение множеств компаний, у которых:  
1) выручка 2023 не NaN; 2) активы 2023 не NaN.  
Выведите список компаний в пересечении.

In [9]:
# Задание 6: Пересечение множеств компаний через Index

# 1. Фильтруем данные только за 2023 год
df_2023 = df[df['year'] == 2023].copy()

# 2. Создаём Index компаний, у которых выручка 2023 не NaN
# Устанавливаем индекс по столбцу 'company' для удобной работы
df_2023_indexed = df_2023.set_index('company')

# companies_revenue — Index компаний с валидной выручкой
companies_revenue = df_2023_indexed[df_2023_indexed['revenue_2110'].notna()].index

# 3. Создаём Index компаний, у которых активы 2023 не NaN
# companies_assets — Index компаний с валидными активами
companies_assets = df_2023_indexed[df_2023_indexed['assets_1600'].notna()].index

# 4. Находим пересечение множеств через метод .intersection()
# Это компании, у которых есть данные и по выручке, и по активам
companies_intersection = companies_revenue.intersection(companies_assets)

# 5. Выводим результат
print("📋 Компании с валидной выручкой 2023:")
print(f"   Количество: {len(companies_revenue)}")
print(f"   Список: {list(companies_revenue)}")

print("\n📋 Компании с валидными активами 2023:")
print(f"   Количество: {len(companies_assets)}")
print(f"   Список: {list(companies_assets)}")

print("\n" + "=" * 60)
print("🔗 Пересечение множеств (есть и выручка, и активы):")
print("=" * 60)
print(f"   Количество компаний: {len(companies_intersection)}")
print(f"\n   Список компаний в пересечении:")
for i, company in enumerate(companies_intersection, 1):
    print(f"     {i}. {company}")

# 6. Дополнительная информация: тип объекта
print(f"\n📊 Тип объекта пересечения: {type(companies_intersection)}")

📋 Компании с валидной выручкой 2023:
   Количество: 7
   Список: ['АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ"', 'АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК"', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1"', 'АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"']

📋 Компании с валидными активами 2023:
   Количество: 7
   Список: ['АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ"', 'АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК"', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ 

### Задание 7
Установите индекс DataFrame как `company` и выберите данные одной компании через `.loc`.  
Возьмите первую компанию из `df['company'].unique()`.

In [10]:
# Задание 7: Установка индекса и выборка через .loc

# 1. Получаем первую уникальную компанию из DataFrame
# df['company'].unique() возвращает массив уникальных названий компаний
# [0] — берём первый элемент этого массива
first_company = df['company'].unique()[0]
print(f"📋 Первая компания: {first_company}")

# 2. Устанавливаем столбец 'company' как индекс DataFrame
# inplace=True — изменяем исходный DataFrame, не создавая копию
df_indexed = df.set_index('company', inplace=False)

# 3. Выбираем все данные для первой компании через .loc
# .loc[имя_индекса] — выборка по значению индекса
company_data = df_indexed.loc[first_company]

# 4. Выводим результат
print("\n" + "=" * 60)
print("📊 Данные компании через .loc:")
print("=" * 60)
print(company_data)

# 5. Дополнительная информация о типе результата
print("\n" + "=" * 60)
print("🔍 Информация о выборке:")
print("=" * 60)
print(f"Тип результата: {type(company_data)}")
print(f"Количество столбцов: {len(company_data)}")
print(f"Названия столбцов: {list(company_data.index)}")

# 6. Показываем первые несколько строк для наглядности
print("\n" + "=" * 60)
print("📋 Первые 5 строк DataFrame с новым индексом:")
print("=" * 60)
print(df_indexed.head())

📋 Первая компания: АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"

📊 Данные компании через .loc:
                                                                                file  \
company                                                                                
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...       Копия 1 Атомэнергопром.xlsx   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...       Копия 1 Атомэнергопром.xlsx   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...  Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...  Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx   

                                                           inn  okved  \
company                                                                 
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...  7706664260  62.09   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...  7706664260  62.09   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...  7706664260  62.09 

### Задание 8
Сделайте MultiIndex по `(company, year)` и отсортируйте индекс.  
Покажите 6 первых строк.

In [11]:
# Задание 8: Создание MultiIndex и сортировка

# 1. Создаём MultiIndex по столбцам 'company' и 'year'
# set_index устанавливает указанные столбцы как индекс
df_multi = df.set_index(['company', 'year'])

# 2. Сортируем индекс (важно для корректной работы с MultiIndex)
# sort_index() сортирует по всем уровням индекса
df_multi = df_multi.sort_index()

# 3. Показываем первые 6 строк
print("📋 Первые 6 строк DataFrame с MultiIndex:")
print("=" * 80)
print(df_multi.head(6))

# 4. Дополнительная информация об индексе
print("\n" + "=" * 80)
print("🔍 Информация о MultiIndex:")
print("=" * 80)
print(f"Тип индекса: {type(df_multi.index)}")
print(f"Уровни индекса: {df_multi.index.names}")
print(f"Количество уровней: {df_multi.index.nlevels}")
print(f"Общее количество строк: {len(df_multi)}")

📋 Первые 6 строк DataFrame с MultiIndex:
                                                                                      file  \
company                                            year                                      
 АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИ... 2022  Копия 3 Газпром_петрозаводск.xlsx   
                                                   2022              Копия 5 Роснефть.xlsx   
                                                   2023  Копия 3 Газпром_петрозаводск.xlsx   
                                                   2023              Копия 5 Роснефть.xlsx   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ... 2022        Копия 1 Атомэнергопром.xlsx   
                                                   2022   Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx   

                                                                inn  okved  \
company                                            year                      
 АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИ...

### Задание 9
Из MultiIndex-таблицы получите срез по году 2023 с помощью `xs`.  
Покажите 5 строк.

In [12]:
# Задание 9: Получение среза по году 2023 через xs()

# 1. Используем xs() для выборки данных по одному уровню MultiIndex
# xs(2023, level='year') — берём все строки где year = 2023
df_2023 = df_multi.xs(2023, level='year')

# 2. Показываем первые 5 строк
print("📋 Данные за 2023 год (через xs):")
print("=" * 80)
print(df_2023.head(5))

# 3. Дополнительная информация
print("\n" + "=" * 80)
print("🔍 Информация о срезе:")
print("=" * 80)
print(f"Количество компаний за 2023 год: {len(df_2023)}")
print(f"Тип индекса после xs: {type(df_2023.index)}")
print(f"Оставшийся уровень индекса: {df_2023.index.name}")

📋 Данные за 2023 год (через xs):
                                                                                 file  \
company                                                                                 
 АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИ...  Копия 3 Газпром_петрозаводск.xlsx   
 АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИ...              Копия 5 Роснефть.xlsx   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...        Копия 1 Атомэнергопром.xlsx   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ...   Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx   
АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО"                                     Копия 7 Славмо.xlsx   

                                                           inn  okved  \
company                                                                 
 АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИ...  1001009551  35.22   
 АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИ...  1001009551  35.22   
АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫ... 

### Задание 10
Заполните пропуски в `margin` медианой по соответствующему году.  
Сделайте это через `groupby('year').transform(...)`. Покажите 10 строк с `margin` и `margin_filled`.

In [13]:
# Задание 10: Заполнение пропусков в margin медианой по году

# 1. Сначала убедимся, что столбец margin существует (из Задания 4)
# Если нет — создаём его
if 'margin' not in df.columns:
    df['margin'] = df['net_profit_2400'] / df['revenue_2110']
    df['margin'] = df['margin'].replace([np.inf, -np.inf], np.nan)

# 2. Вычисляем медиану margin по каждому году с помощью transform
# groupby('year') — группируем по году
# ['margin'] — выбираем столбец margin
# transform('median') — вычисляет медиану для каждой группы и возвращает Series той же длины
median_by_year = df.groupby('year')['margin'].transform('median')

# 3. Заполняем пропуски (NaN) в margin соответствующими медианами
df['margin_filled'] = df['margin'].fillna(median_by_year)

# 4. Показываем 10 строк с обоими столбцами для сравнения
print("📋 Сравнение margin и margin_filled (10 строк):")
print("=" * 80)
display_cols = ['company', 'year', 'margin', 'margin_filled']
print(df[display_cols].head(10).to_string())

# 5. Дополнительная статистика
print("\n" + "=" * 80)
print("🔍 Статистика заполнения:")
print("=" * 80)
print(f"Пропусков в margin до заполнения: {df['margin'].isna().sum()}")
print(f"Пропусков в margin после заполнения: {df['margin_filled'].isna().sum()}")
print(f"\nМедиана margin по годам:")
print(df.groupby('year')['margin'].median().to_string())

📋 Сравнение margin и margin_filled (10 строк):
                                                          company  year    margin  margin_filled
0      АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"  2022       NaN       0.596449
1      АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"  2023  8.576502       8.576502
2  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"  2022       NaN       0.596449
3  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"  2023       NaN       1.120620
4   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"  2022       NaN       0.596449
5   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"  2023       NaN       1.120620
6      ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"  2022  0.274916       0.274916
7      ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"  2023  0.237986       0.237986
8   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"  2022       Na

### Задание 11
Добавьте столбец `okved_section` — первые 2 цифры ОКВЭД (строка), используя `.str.extract` (regex).  
Покажите 5 строк (company, okved, okved_section).

In [14]:
# Задание 11: Добавление столбца okved_section (первые 2 цифры ОКВЭД)

# 1. Используем .str.extract() с регулярным выражением для извлечения первых 2 цифр
# r'^(\d{2})' — регулярное выражение:
#   ^ — начало строки
#   (\d{2}) — захватываем ровно 2 цифры
# expand=False — возвращает Series, а не DataFrame
df['okved_section'] = df['okved'].str.extract(r'^(\d{2})', expand=False)

# 2. Показываем 5 строк с нужными столбцами
print("📋 Первые 5 строк с OKVED и секцией:")
print("=" * 80)
display_cols = ['company', 'okved', 'okved_section']
print(df[display_cols].head(5).to_string())

# 3. Дополнительная статистика по секциям ОКВЭД
print("\n" + "=" * 80)
print("🔍 Распределение по секциям ОКВЭД:")
print("=" * 80)
print(df['okved_section'].value_counts().to_string())

📋 Первые 5 строк с OKVED и секцией:
                                                          company    okved okved_section
0      АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"    62.09            62
1      АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"    62.09            62
2  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"  51.10.1            51
3  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"  51.10.1            51
4   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"    35.22            35

🔍 Распределение по секциям ОКВЭД:
okved_section
35    6
62    4
70    4
51    2
10    2
41    2


### Задание 12
Создайте логический столбец `profit_pos_2023` (True/False) для строк 2023: прибыль > 0.  
Для строк других лет поставьте NaN. Покажите 10 строк.

In [15]:
# Задание 12: Создание столбца profit_pos_2023

# 1. Создаём логический столбец с условиями:
#    - Для 2023 года: True если прибыль > 0, иначе False
#    - Для других лет: NaN
import numpy as np

# Используем np.where для условного присваивания
df['profit_pos_2023'] = np.where(
    df['year'] == 2023,                    # Условие: год = 2023
    df['net_profit_2400'] > 0,             # Если True: проверяем прибыль > 0
    np.nan                                  # Если False: ставим NaN
)

# 2. Показываем 10 строк с нужными столбцами
print("📋 10 строк с profit_pos_2023:")
print("=" * 80)
display_cols = ['company', 'year', 'net_profit_2400', 'profit_pos_2023']
print(df[display_cols].head(10).to_string())

# 3. Дополнительная статистика
print("\n" + "=" * 80)
print("🔍 Статистика по profit_pos_2023:")
print("=" * 80)
print(f"Всего строк: {len(df)}")
print(f"Строк за 2023 год: {(df['year'] == 2023).sum()}")
print(f"Компаний с прибылью > 0 в 2023: {(df['profit_pos_2023'] == True).sum()}")
print(f"Компаний с убытком в 2023: {(df['profit_pos_2023'] == False).sum()}")
print(f"Строк с NaN (не 2023 год): {df['profit_pos_2023'].isna().sum()}")

📋 10 строк с profit_pos_2023:
                                                          company  year  net_profit_2400  profit_pos_2023
0      АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"  2022              NaN              NaN
1      АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"  2023       91493932.0              1.0
2  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"  2022              NaN              NaN
3  ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"  2023              NaN              0.0
4   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"  2022              NaN              NaN
5   АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК"  2023              NaN              0.0
6      ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"  2022      790120077.0              NaN
7      ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ"  2023      655289456.0              1.0
8   АКЦИОНЕРНОЕ 